In [ ]:
from pathlib import Path
import getpass

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import geopandas as gpd

from sentinelhub import (
    SHConfig,
    CRS,
    BBox,
    DataCollection,
    DownloadRequest,
    MimeType,
    MosaickingOrder,
    SentinelHubDownloadClient,
    SentinelHubStatisticalDownloadClient,
    SentinelHubRequest,
    bbox_to_dimensions,
    SentinelHubStatistical,
    Geometry,
    parse_time,
)

In [ ]:
evalscript_raw = """
//VERSION=3
function setup() {
   return {
    input: ["CO"], // This specifies the bands that are looked at
    output: {
      bands: 1,
      // This specifies in which data type the values will be returned
      sampleType: "FLOAT32"
    },
    // Will make a simple mosaic, taking the most recent tiles to fill the bounding box
    mosaicking: "SIMPLE"
  };
}

function evaluatePixel(samples) {
    // Here we could do more calculations which are applied to each pixel,
    // but for now let's just return the value
   return [samples.CO]
}
"""

In [ ]:
from sentinelhub import SHConfig
import getpass

config = SHConfig()

config.sh_client_id = getpass.getpass("Enter your CDSE client ID: ")
config.sh_client_secret = getpass.getpass("Enter your CDSE client secret: ")
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
config.sh_base_url = "https://sh.dataspace.copernicus.eu"

config.save()

In [ ]:
from sentinelhub import SentinelHubSession

try:
    session = SentinelHubSession(config=config)
    print("Authentication successful!")
except Exception as e:
    print(f"Authentication failed: {e}")

In [ ]:
bbox_west_java_banten = BBox([105, -7.90, 108.85, -5.85], crs=CRS.WGS84).transform(CRS(3857))

data_5p = DataCollection.SENTINEL5P.define_from("5p", service_url=config.sh_base_url)

request_raw = SentinelHubRequest(
    evalscript=evalscript_raw,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=data_5p,
            time_interval=("2023-01-01", "2025-04-30"),
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=bbox_west_java_banten,
    resolution=(5500, 3500), 
    config=config,
    data_folder="./data",
)

In [ ]:
import geopandas as gpd

indonesia = gpd.read_file("https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_IDN_2.json")

west_java = indonesia[indonesia['NAME_1'] == 'JawaBarat']
banten = indonesia[indonesia['NAME_1'] == 'Banten']

In [ ]:
Jakarta = indonesia[indonesia["NAME_2"].isin([
    "JakartaBarat", "JakartaPusat", "JakartaSelatan",
    "JakartaTimur", "JakartaUtara"
])]

In [ ]:
capitals = gpd.GeoDataFrame(
    pd.concat([
        west_java[["NAME_2", "geometry"]],
        banten[["NAME_2", "geometry"]],
        Jakarta[["NAME_2","geometry"]]
    ]),
    crs=west_java.crs 
)

In [ ]:
capitals = capitals.to_crs(epsg=3857)

In [ ]:
capitals.count()

In [ ]:
evalscript_stat = """
//VERSION=3
function setup() {
    return {
        input: ["CO", "dataMask"],
        output: [{
          id: "default",
          bands: ["CO"],
          sampleType: "FLOAT32"
        },
        {
          id: "dataMask",
          bands: 1,
        }],
        mosaicking: "ORBIT"
    };
}

function isClear(sample) {
    return sample.dataMask == 1;
}

function sum(array) {
    let sum = 0;
    for (let i = 0; i < array.length; i++) {
        sum += array[i].CO;
    }
    return sum;
}

function evaluatePixel(samples) {
    const clearTs = samples.filter(isClear)
    const mean = sum(clearTs) / clearTs.length
    return {default: [mean], dataMask: [clearTs.length]}
}
"""

In [ ]:
aggregation = SentinelHubStatistical.aggregation(
    evalscript=evalscript_stat,
    time_interval=("2023-05-01", "2025-04-30"),
    aggregation_interval="P1D",
    size=(1, 1)
)

input_data = SentinelHubStatistical.input_data(
    DataCollection.SENTINEL5P.define_from("5p", service_url=config.sh_base_url)
)

requests = []

In [ ]:
for geo_shape in capitals.geometry.values:
    request = SentinelHubStatistical(
        aggregation=aggregation,
        input_data=[input_data],
        geometry=Geometry(geo_shape, crs=CRS(capitals.crs)),
        config=config,
    )
    requests.append(request)

download_requests = [request.download_list[0] for request in requests]
client = SentinelHubStatisticalDownloadClient(config=config)
pollution_stats = client.download(download_requests, max_threads=5, show_progress=True)

In [ ]:
valid_districts = [city for city, stats in zip(capitals["NAME_2"], pollution_stats)
                  if any(d["outputs"]["default"]["bands"]["CO"]["stats"]["sampleCount"] > 0
                       for d in stats["data"])]
print(f"Districts with data: {len(valid_districts)}")

In [ ]:
def stats_to_df(stats_data, geometries, keep_all=False):
    df_data = []
    for idx, (single_data, geom) in enumerate(zip(stats_data["data"], geometries)):
        df_entry = {
            "x_coord": geom.centroid.x,
            "y_coord": geom.centroid.y,
            "interval_from": parse_time(single_data["interval"]["from"]).date(),
            "interval_to": parse_time(single_data["interval"]["to"]).date(),
            "has_data": False  
        }

        for output_name, output_data in single_data["outputs"].items():
            for band_name, band_values in output_data["bands"].items():
                band_stats = band_values["stats"]
                df_entry["has_data"] = band_stats["sampleCount"] > 0

                for stat_name, value in band_stats.items():
                    col_name = f"{output_name}_{band_name}_{stat_name}"
                    if stat_name == "percentiles":
                        for perc, perc_val in value.items():
                            df_entry[f"{col_name}_{perc}"] = perc_val
                    else:
                        df_entry[col_name] = value

        df_data.append(df_entry)

    return pd.DataFrame(df_data)

co_dfs = [stats_to_df(polygon_stats, [geom] * len(polygon_stats["data"])).assign(city=city_name)
         for (polygon_stats, geom, city_name) in zip(pollution_stats, capitals.geometry.values, capitals["NAME_2"])]

co_df = pd.concat(co_dfs, ignore_index=True)

co_df["interval_from"] = pd.to_datetime(co_df["interval_from"])

co_df["month"] = co_df["interval_from"].dt.month
co_df["month_name"] = co_df["interval_from"].dt.month_name()

co_df.to_csv("./co_2022.csv")

In [ ]:
co_df = pd.read_csv("./co_2022.csv",index_col=0)

In [ ]:
co_df= co_df.sort_values(by="interval_from")

In [ ]:
co_df["location"] = co_df.apply(lambda row: f"{row['x_coord']:.2f},{row['y_coord']:.2f}", axis=1)

In [ ]:
gdf = gpd.GeoDataFrame(
    co_df,
    geometry=gpd.points_from_xy(co_df["x_coord"], co_df["y_coord"]),
    crs="EPSG:3857"  
)

gdf = gdf.to_crs("EPSG:4326")

gdf["x_coord"] = gdf.geometry.x
gdf["y_coord"] = gdf.geometry.y

gdf = gdf.drop(columns="geometry")

co_df = gdf

In [ ]:
print(co_df["city"].nunique())


In [ ]:
co_df.to_csv("./co_cleaned.csv")

In [ ]:
heatmap_df = co_df.pivot(index="city", columns="interval_from", values="default_CO_mean")

In [ ]:
import plotly.express as px

fig = px.imshow(
    heatmap_df,
    labels=dict(x="Date", y="City", color="CO Level"),
    aspect="auto",
    color_continuous_scale="Viridis"
)

fig.update_layout(
    title="Interactive CO Heatmap Over Time [mol/m^2]",
    xaxis_nticks=40,
    height=1000,
    width=1000,
    margin=dict(t=50, b=50, l=100, r=50),
)

fig.update_yaxes(autorange="reversed")

fig.show()


In [ ]:
# %%
import pandas as pd
from google.colab import drive
import numpy as np
import pandas as pd
import matplotlib.dates as mdates 


import matplotlib.pyplot as plt
import time
import datetime

df = pd.read_csv("/content/drive/MyDrive/TA/Dataset/co.csv")

df["date"] = pd.to_datetime(df["date"])

cities = df["city"].unique()

n_cities = len(cities)
n_cols = 4  
n_rows = (n_cities + n_cols - 1) // n_cols 

fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 4 * n_rows), squeeze=False) 

axes = axes.flatten()

for i, city in enumerate(cities):
    city_df = df[df["city"] == city].copy()

    city_df = city_df.sort_values(by="date")

    ax = axes[i]
    ax.plot(city_df["date"], city_df["CO_mean"], marker="o")
    ax.set_xlabel("Tanggal")
    ax.set_ylabel("Rata-rata CO")
    ax.set_title(f"Rata-rata CO di {city}")
    ax.grid(True)

    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout() 
plt.show()

In [ ]:
import pandas as pd
from google.colab import drive
import numpy as np
import pandas as pd


drive.mount('/content/drive/', force_remount=True)

dir = '/content/drive/My Drive/TA/'
import sys
sys.path.insert(0, dir)

import matplotlib.pyplot as plt
import time
import datetime

df = pd.read_csv("/content/drive/MyDrive/TA/Dataset/Sea/CO_Sea_4xGrid.csv")

df["date"] = pd.to_datetime(df["date"])


In [ ]:
df= df.rename(columns={"mean":"MeanCO"})

In [ ]:
df.describe()

In [ ]:
unique_locations = df[["longitude", "latitude"]].drop_duplicates().reset_index(drop=True)
display(unique_locations)

In [ ]:
df.isna().sum()

In [ ]:
df_clean = df.dropna().reset_index(drop=True)
df_clean = df_clean[df_clean.MeanCO>=0]

In [ ]:
df_clean = df_clean[df_clean.MeanCO>=0]

In [ ]:
df_clean.describe()

In [ ]:
df_agg= df.groupby("date").aggregate("mean")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5)) 

ax.plot(df_agg.MeanCO,marker="o")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Rata-rata CO")
ax.set_title("Rata-rata CO Area Laut Teritorial")
ax.grid(True)
plt.show()

In [ ]:
df_clean.to_csv("/content/drive/MyDrive/TA/Dataset/Sea/CO_Sea_cleaned.csv")

In [ ]:
unique_locations = df_clean[["longitude", "latitude"]].drop_duplicates().reset_index(drop=True)
display(unique_locations)